# 第 14 章 外部効果とコースの定理: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/micro-basics-programming/14-externality-and-coase.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
demand_intercept = 100  # 逆需要 P = a − b·Q の切片 a
demand_slope = 1        # 逆需要の傾き b
cost_intercept = 10     # 私的な限界費用 MC = c + d·Q の切片 c
cost_slope = 1          # 私的な限界費用の傾き d
external_cost = 20      # 1 単位あたりの外部費用（負なら外部便益）
q_max = 100             # 図の数量の範囲
p_max = 100             # 図の価格の範囲
# コースの定理: 加害者の利益と、被害者の被害
gain = 100              # 汚染する加害者の利益
damage = 60             # 被害者の被害
victim_has_right = True  # 権利が、被害者にあるか（False なら加害者）
# 排出権取引: 企業ごとの、削減費用の傾き（大きいほど、削減が苦しい）
slopes = [1, 2, 4]
total_reduction = 14    # 全体で削減する量

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 外部効果とピグー税

In [ ]:
from lib.chapter14.externality import (
    ExternalityMarket,
    externality_deadweight_loss,
    market_quantity,
    pigou_tax,
    quantity_with_tax,
    socially_optimal_quantity,
)

market = ExternalityMarket(
    demand_intercept, demand_slope, cost_intercept, cost_slope, external_cost
)
tax = pigou_tax(market)
print(f"市場の数量: {market_quantity(market):.2f}、"
      f"社会的に最適な数量: {socially_optimal_quantity(market):.2f}")
print(f"ピグー税: {tax:.2f}、税のもとの数量: {quantity_with_tax(market, tax):.2f}")
print(f"死荷重: {externality_deadweight_loss(market):.2f}")

## コースの定理と排出権取引

In [ ]:
from lib.chapter14.externality import (
    abatement_cost,
    coase_outcome,
    emissions_trading,
    uniform_reduction,
)

result = coase_outcome(gain, damage, victim_has_right)
print(f"コース（利益 {gain}、被害 {damage}）: 汚染する {result.pollution}、"
      f"払う金額の範囲 {result.payment_range}")
price, reductions = emissions_trading(slopes, total_reduction)
print(f"排出権の価格 {price:.2f}、削減量 {[round(r, 4) for r in reductions]}、"
      f"費用 {abatement_cost(slopes, reductions):.4f}")
uniform = uniform_reduction(slopes, total_reduction)
print(f"一律規制の削減量 {[round(r, 4) for r in uniform]}、"
      f"費用 {abatement_cost(slopes, uniform):.4f}")

## 図

In [ ]:
from lib.chapter14.plot import plot_externality

plot_externality(market, q_max, p_max)

## 試してみよう

- `external_cost = 40` にすると、最適な数量は 25、ピグー税は 40、死荷重は 400.0 です。
  外部費用が 2 倍になると、死荷重は **4 倍**になります（2 乗に比例）
- `external_cost = 0` にすると、市場の数量と最適な数量が一致して（45）、
  死荷重は 0 です。`external_cost = -10`（外部便益）にすると、最適な数量は 50、
  税は −10（補助金）、死荷重は 25.0 です
- `gain = 40` にすると、権利が被害者にあっても、加害者にあっても、汚染しません。
  `gain = 100` なら、どちらでも、汚染します。権利の持ち主が違っても、**汚染するかは
  同じ**で、お金の流れだけが変わります
- `slopes = [1, 1, 1]` にすると、排出権取引も一律規制も、費用は約 32.67 で同じです。
  `slopes = [1, 1, 4]` にすると、約 43.56 と約 65.33 で、費用の差が開きます。
  企業の費用の差が大きいほど、取引の利益が大きくなります